# B2.2 · Reliability and cost under non-determinism

**Function B — Application Security with an AI SDLC → Trusting the Harness that Tests CyberTravels**  ·  *AI for Security*

Builds on **[B2.1 · Evaluating a security harness](https://spbreed.github.io/cyber-commons/lessons/B2.1.html)**.

| | |
|---|---|
| Tools used | Inspect, promptfoo, CyberGym, GLM-4.6, Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

At 80% per-run reliability a harness is 99.9% reliable with a human picking the good answer, and 33% reliable unattended. Both numbers are true. Quoting the first for a system that runs unattended is where most harness claims quietly go wrong.

> **At CyberTravels.** At 80% per-run reliability, a review harness left to run unattended over CyberTravels' releases is right about a third of the time — and somebody at CyberTravels is paying per confirmed finding.

## 2 · The framework

```
   per-run reliability 80%

   pass@5  = at least one of five worked   -> 99.97%   (a human picks)
   pass^5  = all five worked               -> 32.8%    (unattended)

   both numbers are true. the honest one depends on who is watching.

   plus the two nobody computes: $ per confirmed finding,
   and analyst minutes per accepted finding
```

A single run tells you almost nothing about a stochastic system, and almost
every published harness result is a single run.

Two metrics, and the gap between them is the whole lesson:

**pass@k** — succeeded *at least once* in k attempts. This is the right metric
when you can cheaply check which attempt was correct and keep it. A code
generator whose output you compile and test is a pass@k system: five tries and
one good answer is a good day.

**pass^k** — succeeded *every time*, k out of k. This is the right metric when
the run is autonomous and nobody is checking, which describes every security
harness that files tickets, gates a merge or closes an alert.

A harness at 80% per-run reliability has a pass@5 of 99.97% and a pass^5 of
33%. Both numbers are true. Quoting the first one for a system that runs
unattended is where most harness claims quietly go wrong.

The other half is **run-to-run variance.** If the same input produces eleven
findings one day and four the next, the average is not a description of
anything, and any change you make afterwards is being measured against noise.

Reliability is one of three numbers that decide whether a harness is worth
running, and most teams measure only the first:

- **Accuracy**, usually against whatever the tool happened to find, which is
  circular. A **seeded corpus** of deliberately planted defects turns recall
  into a fraction with a real denominator.
- **Money**, per unit of value. Total spend is on the invoice; *cost per
  confirmed finding* is what changes when someone switches model tier.
- **Human time**, which is almost never measured and decides whether the tool
  survives. Four hundred findings a week at nine minutes each has not removed
  work. It has moved the work, renamed it triage, and made it somebody else's.

## 3 · The two metrics, from the same runs

In [ ]:
import random

def run_once(reliability, rng):
    return rng.random() < reliability

def measure(reliability, k=5, trials=2000, seed=3):
    rng = random.Random(seed)                  # seeded: identical every run
    at_least_one = every_time = 0
    for _ in range(trials):
        outcomes = [run_once(reliability, rng) for _ in range(k)]
        at_least_one += any(outcomes)
        every_time   += all(outcomes)
    return at_least_one / trials, every_time / trials

print(f"{'per-run':>9}{'pass@5':>10}{'pass^5':>10}  what it means unattended")
for r in (0.95, 0.90, 0.80, 0.60, 0.50):
    at_k, pow_k = measure(r)
    note = ("dependable" if pow_k > .8 else
            "coin flip" if pow_k > .3 else "fails most nights")
    print(f"{r:>8.0%}{at_k:>10.1%}{pow_k:>10.1%}  {note}")
print()
print("At 80% per-run the same harness is 99.9% reliable if a human picks the")
print("good answer, and 33% reliable if nobody is looking.")

## 4 · Where it breaks — the demo that was a lucky run

In [ ]:
def one_demo(reliability, seed):
    return run_once(reliability, random.Random(seed))

demos = [one_demo(0.6, s) for s in range(12)]
print("twelve single-run demos of the same 60% harness:")
print("   " + " ".join("PASS" if d else "fail" for d in demos))
print(f"   -> {demos.count(True)} passed")
print()
print("Publish any of the passes. Every one is an honest single run. None of")
print("them is a measurement, and the reader has no way to tell which they got.")
assert demos.count(True) and demos.count(False)

## 5 · Variance, and why it precedes every other question

In [ ]:
def findings_per_run(base, noise, rng):
    return max(0, int(rng.gauss(base, noise)))

def variance_profile(base, noise, runs=30, seed=11):
    rng = random.Random(seed)
    xs = [findings_per_run(base, noise, rng) for _ in range(runs)]
    mean = sum(xs) / len(xs)
    sd = (sum((x - mean) ** 2 for x in xs) / len(xs)) ** 0.5
    return xs, mean, sd

for noise in (0.5, 3.0):
    xs, mean, sd = variance_profile(8, noise)
    print(f"noise sd={noise}:  mean {mean:.1f}  sd {sd:.2f}  "
          f"range {min(xs)}-{max(xs)}")
    print(f"   runs: {xs[:12]} ...")

_, m_stable, sd_stable = variance_profile(8, 0.5)
_, m_noisy,  sd_noisy   = variance_profile(8, 3.0)
improvement = 1.5
print()
print(f"suppose a change adds {improvement} findings on average.")
print(f"   against sd {sd_stable:.2f}: visible after a handful of runs")
print(f"   against sd {sd_noisy:.2f}: indistinguishable from a quiet Tuesday")
print()
print("Until variance is characterised, every A/B comparison you run is")
print("measuring the dice.")
assert sd_noisy > sd_stable

## 6 · The control — separate harness failure from model failure

Before changing either one, find out which is moving.

In [ ]:
def attribute(runs_same_model_same_harness, runs_same_model_new_harness):
    """If output changes when only the harness changed, it was the harness."""
    a = sum(runs_same_model_same_harness) / len(runs_same_model_same_harness)
    b = sum(runs_same_model_new_harness) / len(runs_same_model_new_harness)
    return a, b, ("harness" if abs(b - a) > 0.15 else "not the harness")

rng = random.Random(5)
baseline = [run_once(0.6, rng) for _ in range(200)]
better_harness = [run_once(0.85, rng) for _ in range(200)]   # same model, better verifier
a, b, verdict = attribute(baseline, better_harness)
print(f"same model, original harness : {a:.0%}")
print(f"same model, better verifier  : {b:.0%}")
print(f"attribution                  : {verdict}")
print()
print("Twenty-five points of reliability, no model change. Teams routinely")
print("spend that budget on a bigger backbone instead, because the harness")
print("was never measured separately.")
assert verdict == "harness"

## 7 · Verify — the scorecard line that has to be published

In [ ]:
k = 5
rel = 0.85
at_k, pow_k = measure(rel, k=k)
xs, mean, sd = variance_profile(8, 0.9)
card = {
  "per_run_reliability": rel,
  "k": k,
  "pass_at_k": round(at_k, 4),
  "pass_pow_k": round(pow_k, 4),
  "runs_measured": 2000,
  "findings_mean": round(mean, 2),
  "findings_sd": round(sd, 2),
  "seed": 3,
  "unattended": True,
  "headline_metric": "pass^k",       # because unattended is True
}
for kk, vv in card.items():
    print(f"   {kk:22s}{vv}")
print()
print("The headline metric is chosen by how the harness runs, not by which")
print("number is larger. An unattended harness that reports pass@k is")
print("reporting the reliability of a system it is not.")
assert card["headline_metric"] == "pass^k" and card["pass_pow_k"] < card["pass_at_k"]

## 8 · The other two numbers — money, and somebody's afternoon

Reliability says whether the harness can be left alone. Cost per confirmed finding and analyst minutes per accepted finding say whether leaving it alone is worth doing. Both need a corpus whose defects you planted, so recall has a real denominator.

In [ ]:
SEEDED_CORPUS = {
 f"unit_{i:02d}": (i % 4 == 0, ["CWE-22", "CWE-78", "CWE-89", "CWE-79"][i % 4])
 for i in range(40)
}
planted = sorted(u for u, (is_bug, _) in SEEDED_CORPUS.items() if is_bug)
print(f"units in corpus : {len(SEEDED_CORPUS)}")
print(f"planted defects : {len(planted)}   <- the denominator is now a fact")

def harness_run(corpus, sensitivity, seed=2):
    """Higher sensitivity finds more real defects, and more false ones."""
    rng = random.Random(seed)
    out = []
    for unit, (is_bug, cwe) in sorted(corpus.items()):
        if is_bug and rng.random() < sensitivity:
            out.append((unit, cwe, True))
        elif not is_bug and rng.random() < sensitivity * 0.35:
            out.append((unit, cwe, False))
    return out

def scorecard(found, corpus, tokens_per_unit=1800, usd_per_1k=0.002,
              analyst_minutes=9):
    tp = [f for f in found if f[2]]
    planted_n = sum(1 for _, (b, _) in corpus.items() if b)
    spend = len(corpus) * tokens_per_unit / 1000 * usd_per_1k
    return {"recall": len(tp) / planted_n,
            "precision": len(tp) / len(found) if found else 0.0,
            "usd_per_finding": spend / len(tp) if tp else None,
            "analyst_minutes": len(found) * analyst_minutes,
            "minutes_per_accepted": len(found) * analyst_minutes / len(tp) if tp else None}

MANUAL_MINUTES = 40 * 4          # a human reading the same forty units
print(f"\n{'sens':>6}{'recall':>9}{'prec':>7}{'$/find':>9}{'min/accepted':>14}"
      f"{'review load':>13}")
for s in (0.4, 0.7, 0.95):
    c = scorecard(harness_run(SEEDED_CORPUS, s), SEEDED_CORPUS)
    verdict = "saves time" if c["analyst_minutes"] < MANUAL_MINUTES else "COSTS MORE"
    print(f"{s:>6.2f}{c['recall']:>9.0%}{c['precision']:>7.0%}"
          f"{c['usd_per_finding']:>9.3f}{c['minutes_per_accepted']:>14.1f}"
          f"{verdict:>13}")

best_recall = scorecard(harness_run(SEEDED_CORPUS, 0.95), SEEDED_CORPUS)
print()
print(f"At the highest sensitivity recall is {best_recall['recall']:.0%} and the review")
print(f"queue is {best_recall['analyst_minutes']} minutes against {MANUAL_MINUTES} for reading the code by")
print("hand. The accuracy metric improved and the thing got worse - which stays")
print("invisible unless review load is a first-class number beside it.")
assert best_recall["analyst_minutes"] > MANUAL_MINUTES

## What you just proved

pass@5 and pass^5 are computed from the same 2000 trials and diverge sharply: at 80% per-run reliability the harness is 99.9% reliable with a human picking the good answer and 33% reliable unattended. Twelve single-run demos of a 60% harness return a mix of passes and failures, and a change worth 1.5 findings is shown to be invisible against a standard deviation of 3. On the same seeded corpus of 40 units with 10 planted defects, raising sensitivity from 0.70 to 0.95 lifts recall from 60% to 90% and pushes the review queue from 'saves time' to 180 analyst minutes against 160 for reading the code by hand.

## Your turn

Run your harness on one fixed task five times and count how many times it fully succeeded. That integer, out of five, is the number to put in front of anyone deciding whether to let it run unattended. Then divide last month's spend by the findings anyone actually accepted, and compare that to an hour of the reviewer's time.

---

**Next → [B2.3 · Honeypots, canaries and deception in the agent's environment](https://spbreed.github.io/cyber-commons/lessons/B2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*